In [29]:
from random import randint
import numpy as np
# Define the parameters of the Hamming code
n = 7
k = 4
moc = 7
m = 3
r = 4
#systematic form for easier processing
H = np.array([
    [1, 0, 0, 1, 1, 0, 1],
    [0, 1, 0, 1, 0, 1, 1],
    [0, 0, 1, 0, 1, 1, 1]
])

G = np.array([
    [1, 1, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 1, 0, 0],
    [0, 1, 1, 0, 0, 1, 0],
    [1, 1, 1, 0, 0, 0, 1]
])

print(np.mod(H@np.transpose(G),2))

[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]


In [30]:
#now lets make redundant rows for H, just generate all possible checks with H
M = np.array([[0,1,1],
                [1,0,1],
                [1,1,0],
                [1,1,1]])

meta_checks = np.mod(M@H,2)
Hoc = np.concatenate((H,meta_checks),axis=0)
print(meta_checks)
print(Hoc)

[[0 1 1 1 1 0 0]
 [1 0 1 1 0 1 0]
 [1 1 0 0 1 1 0]
 [1 1 1 0 0 0 1]]
[[1 0 0 1 1 0 1]
 [0 1 0 1 0 1 1]
 [0 0 1 0 1 1 1]
 [0 1 1 1 1 0 0]
 [1 0 1 1 0 1 0]
 [1 1 0 0 1 1 0]
 [1 1 1 0 0 0 1]]


now we consider two cases

case 1: we only measure the three generators in H, we have

$z = He^{\mathsf{T}}+e_z $

where
$He^{\mathsf{T}}:=z_c$ is the "correct" syndrome when no measurement errors occur
$e_z$ is the measurement error,
$z$ is the observed syndrome

Known: $z$ and $H$

Unknown: $e$ and $e_z$

As we only use three generators, $H$ is full rank here. We cannot solve $e$ and $e_z$ because for every $e$, there is a solution for $e_z$




In data-syndrome (DS) code term, we can define the check matrix $(H\quad I_m)$ such that
$z = \begin{pmatrix} H & I_m\end{pmatrix}\begin{pmatrix} e \\ e_z\end{pmatrix}$
Since $H$ is full rank, $(H \quad I)$ is also full rank, so the system of linear equations always has solution, and not unique. In fact, for any $e$, there is always a $e_z$, so we cannot deteremine the syndrome errors at all

$\Rightarrow$ we need redundant syndrome bits

Case 2: we measrue also the meta checks generated with $MH$, we have

$z_{oc} = \begin{pmatrix} H \\ MH\end{pmatrix} e^{\mathsf{T}} + e_{z,oc}=\begin{pmatrix} z_c \\ M z_c\end{pmatrix} + \begin{pmatrix} e_z\\ e_{z}'\end{pmatrix}$

this is equavalent as saying

$z_{oc} = (z \quad z')^{\mathsf{T}}= \begin{pmatrix} H & I_m &0\\ MH &0&I_r\end{pmatrix} \begin{pmatrix} e^{\mathsf{T}} \\ e_z^{\mathsf{T}}  \\ e_z'^{\mathsf{T}}  \\ \end{pmatrix} $

applying the same row operations on both sides, we have

$\begin{pmatrix} z \\ Mz + z'\end{pmatrix}=\begin{pmatrix} H & I_m &0\\ 0&M&I_r\end{pmatrix} \begin{pmatrix} e^{\mathsf{T}} \\ e_z^{\mathsf{T}}  \\ e_z'^{\mathsf{T}}  \\ \end{pmatrix}$, where we only need to solve the error vector


In [53]:
#now we implement case 2
#in our case, we don't need to go solve M as we deliberately choose M

#first sample errors on both data bits and syndrome bits, for simplicity consider XZ channel, and consider only X error
#in complete implement, use quaternary BP and depolarizing noise
pd = 0.2
ps = 0.2
e = np.zeros([1,n])
ez = np.zeros([1,n])
sample_method = 1

if sample_method == 1:
    for i in np.array(range(n)):
        if np.random.rand() < pd:
            e[0,i] = 1
        else:
            e[0,i] = 0

    for i in np.array(range(moc)):
        if np.random.rand() < ps:
            ez[0,i] = 1
        else:
            ez[0,i] = 0
else:
    e = np.array([[1,1,0,0,0,0,0]])
    ez = np.array([[0,0,0,1,0,0,0]])

zoc = np.transpose(np.mod(np.mod(Hoc@np.transpose(e),2) + np.transpose(ez),2))
print('data error e:', e)
print('syndrome error ez:', ez)
print('complete measured syndrome zoc', zoc)

print(m)
z = zoc[0,:m]
z_meta = zoc[0, m:]

print('syndrome from data error z:', z)
print('syndrome from data error after M:', z_meta)

Mz = np.mod(M@z,2)
print('meta syndrome z_meta:', Mz)

data error e: [[0. 0. 0. 0. 1. 0. 1.]]
syndrome error ez: [[1. 0. 0. 0. 0. 1. 0.]]
complete measured syndrome zoc [[1. 1. 0. 1. 0. 0. 1.]]
3
syndrome from data error z: [1. 1. 0.]
syndrome from data error after M: [1. 0. 0. 1.]
meta syndrome z_meta: [1. 1. 0. 0.]


In [54]:
#now we can build a linear equation system b = Ax, where b is the observed syndrome, A is the extended check matrix, and x is the error vector we want to solve for
b = np.concatenate((z,np.mod(Mz+z_meta,2)),axis=0)
A = np.concatenate((
    np.concatenate((H, np.eye(m), np.zeros((m, r))), axis=1),
    np.concatenate((np.zeros((r, n)), M, np.eye(r)), axis=1)
), axis=0)
print('b:', b)
print('A:', A)

b: [1. 1. 0. 0. 1. 0. 1.]
A: [[1. 0. 0. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 1. 0. 1. 1. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 1.]]


In [55]:
import numpy as np

def solve_gf2(A, b):
    """
    Solve A x = b over GF(2).

    Returns one solution x if it exists.
    Free variables, if any, are set to 0.
    Raises ValueError if no solution exists.
    """
    A = np.asarray(A, dtype=np.uint8) & 1
    b = np.asarray(b, dtype=np.uint8).reshape(-1, 1) & 1

    m, n = A.shape
    Ab = np.concatenate([A, b], axis=1)

    row = 0
    pivots = []

    for col in range(n):
        # Find pivot row
        pivot_rows = np.where(Ab[row:, col] == 1)[0]
        if len(pivot_rows) == 0:
            continue

        pivot = pivot_rows[0] + row

        # Swap pivot row into place
        Ab[[row, pivot]] = Ab[[pivot, row]]

        # Eliminate this column from all other rows using XOR
        for r in range(m):
            if r != row and Ab[r, col]:
                Ab[r] ^= Ab[row]

        pivots.append(col)
        row += 1

        if row == m:
            break

    # Check for inconsistency: 0 = 1
    for r in range(m):
        if not Ab[r, :n].any() and Ab[r, n]:
            raise ValueError("No solution exists over GF(2)")

    # Read off one solution
    x = np.zeros(n, dtype=np.uint8)
    for r, col in enumerate(pivots):
        x[col] = Ab[r, n]

    return x

In [56]:
x = solve_gf2(A, b)
print(x)
print((np.array(A) @ x) % 2)

#now cut the vector into different errors
e_hat = x[:n]
e_z_hat = x[n:]
print('estimated data error:', e_hat)
print('estimated syndrome error:', e_z_hat)
print('estimated syndrome:', np.mod(e_z_hat + zoc[0,],2))
print('computed syndrome from the estimates', np.mod(Hoc@np.transpose(e_hat)+ e_z_hat,2))
print('complete measured syndrome zoc', zoc[0].astype(int))

[1 1 1 0 0 0 0 0 0 1 1 0 0 0]
[1. 1. 0. 0. 1. 0. 1.]
estimated data error: [1 1 1 0 0 0 0]
estimated syndrome error: [0 0 1 1 0 0 0]
estimated syndrome: [1. 1. 1. 0. 0. 0. 1.]
computed syndrome from the estimates [1 1 0 1 0 0 1]
complete measured syndrome zoc [1 1 0 1 0 0 1]


we see that for any combination of data and syndrome error, we can find estimates such that all the observed syndromes are consistent with the estimates, but the estimates can be very different from the actual errors. This is because we have more unknowns than equations, so we cannot uniquely determine the errors. However, if we have some prior knowledge about the error distribution, we can use that to find the most likely error pattern among all the consistent solutions. This is where the belief propagation (BP) algorithm comes in, which can efficiently find the most likely error pattern given the observed syndromes and the error model.